# UR5e pi0-FAST LoRA Fine-Tuning

**All cells run in Google Colab Pro (A100 recommended) unless marked `[LAPTOP]`.**

## Requirements
- Google Colab Pro with A100 GPU (Runtime → Change runtime type → A100)
- HuggingFace account with a **write** token (`sheilsarda/pi0_ur5_fast_v1`, `sheilsarda/pi0_ur5_base_v1`)
- W&B account (wandb.ai)
- Dataset already on HuggingFace Hub: `sheilsarda/ur5_isaac_sim_v1`

## Current state
- `pi0_ur5_fast_v1`: 10k steps trained, checkpoint on HF Hub
- `pi0_ur5_base_v1`: not yet trained

In [ ]:
# [LAPTOP] — run this from your laptop, not Colab
#
# Prerequisites:
#   cd ~/Development/openpi && source .venv/bin/activate
#   huggingface-cli login   (paste your HF write token)
#
# Then run this cell, or paste the equivalent into a terminal:

import subprocess
result = subprocess.run([
    "python3", "-c",
    """
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
dataset = LeRobotDataset('sheilsarda/ur5_isaac_sim_v1')
dataset.push_to_hub(
    tags=['ur5e', 'isaac-sim'],
    private=False,
    push_videos=True,
    license='apache-2.0',
)
print('Done! Dataset live at https://huggingface.co/datasets/sheilsarda/ur5_isaac_sim_v1')
"""
], capture_output=False)


In [ ]:
# Cell 1: Verify GPU
!nvidia-smi

In [ ]:
# Cell 2: Clone repo and install dependencies
!git clone https://github.com/sheilsarda/openpi.git
%cd openpi
!pip install uv -q
!uv sync

In [ ]:
# Cell 3: Authenticate HuggingFace (needed to download dataset AND upload checkpoints)
from huggingface_hub import login

# IMPORTANT: You must use a WRITE token to upload checkpoints later.
# Get your token from: https://huggingface.co/settings/tokens
login()

In [ ]:
# Cell 4: Authenticate W&B
import wandb
wandb.login()  # paste your W&B API key

In [ ]:
# Cell 4b: Authenticate Google Cloud
# Required to download pretrained weights from gs://openpi-assets/
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Cell 5: Mount Google Drive — checkpoints will be saved here
# so they survive the Colab session ending
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/openpi_checkpoints', exist_ok=True)

# Symlink so openpi writes checkpoints directly to Drive
if not os.path.exists('/content/openpi/checkpoints'):
    os.symlink('/content/drive/MyDrive/openpi_checkpoints', '/content/openpi/checkpoints')

print('Checkpoints will be saved to: /content/drive/MyDrive/openpi_checkpoints')

---
## Head-to-Head: pi0-FAST vs pi0-base

Run one section at a time (or both sequentially). Both use LoRA, batch_size=16, 10k steps.

In [ ]:
# Cell 5: Mount Google Drive — checkpoints will be saved here
# so they survive the Colab session ending.
# NOTE: Run the clone cell (Cell 2) before this one.
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/openpi_checkpoints', exist_ok=True)

# Symlink so openpi writes checkpoints directly to Drive.
# os.symlink(src, dst): src=real path on Drive, dst=path inside the repo.
if not os.path.islink('/content/openpi/checkpoints') and not os.path.exists('/content/openpi/checkpoints'):
    os.symlink('/content/drive/MyDrive/openpi_checkpoints', '/content/openpi/checkpoints')

print('Checkpoints dir:', os.path.realpath('/content/openpi/checkpoints'))

---
## Training

Each model has three cells:
1. **Norm stats** — run once per model config (or after any dataset change); fast, CPU-only
2. **Train: FROM SCRATCH** — wipes existing checkpoints, starts from step 0
3. **Train: RESUME** — continues from the latest checkpoint; `--num-train-steps` is the new total

Run **one** of cells 2 or 3, not both.

---
### pi0-FAST (`pi0_ur5`)

In [ ]:
# Norm stats — pi0-FAST
# Skip if already computed and the dataset hasn't changed.
!uv run scripts/compute_norm_stats.py --config-name=pi0_ur5

In [ ]:
# Train pi0-FAST — RESUME
# Continues from the latest checkpoint in checkpoints/pi0_ur5/ur5_fast_v1/.
# --num-train-steps is the TOTAL target (e.g. already at 10k → set 20k to run 10k more).
# W&B: resumes the existing ur5_fast_v1 run automatically.
!uv run scripts/train.py pi0_ur5 \
  --exp-name ur5_fast_v1 \
  --resume \
  --num-train-steps 20000

In [ ]:
# Upload pi0-FAST checkpoint to HuggingFace
# Uploads the full checkpoints/pi0_ur5/ur5_fast_v1/ folder (all saved steps).
from huggingface_hub import HfApi
import os

api = HfApi()
repo_id = 'sheilsarda/pi0_ur5_fast_v1'
api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)

# Infer the latest step from the checkpoint directory for the commit message.
ckpt_dir = '/content/drive/MyDrive/openpi_checkpoints/pi0_ur5/ur5_fast_v1'
steps = sorted(int(d) for d in os.listdir(ckpt_dir) if d.isdigit())
latest_step = steps[-1] if steps else '?'

api.upload_folder(
    folder_path=ckpt_dir,
    repo_id=repo_id,
    repo_type='model',
    commit_message=f'pi0-FAST LoRA ur5 checkpoint step {latest_step}',
)
print(f'Uploaded step {latest_step} → https://huggingface.co/{repo_id}')

# [LAPTOP] Download pi0-FAST checkpoint from HuggingFace for local serving


In [ ]:
#
# Prerequisites (run once):
#   cd ~/Development/openpi && source .venv/bin/activate
#   pip install huggingface_hub
#   huggingface-cli login   (read token is sufficient)
#
# Downloads only the step-15000 checkpoint subdirectory.

import logging
import sys
from huggingface_hub import snapshot_download

# Enable logging to see more details from the huggingface_hub library
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

print("Starting download...")

local_dir = '/home/sheil/Development/openpi/checkpoints/pi0_ur5/ur5_fast_v1'

path = snapshot_download(
    repo_id='sheilsarda/pi0_ur5_fast_v1',
    repo_type='model',
    local_dir=local_dir,
    allow_patterns='15000/**',
)

print(f"Download complete. Model saved to: {path}")
print()
print('To serve step 15000:')
print('  cd ~/Development/openpi')
print('  uv run scripts/serve_policy.py policy:checkpoint \\')
print('      --policy.config=pi0_ur5 \\')
print('      --policy.dir=checkpoints/pi0_ur5/ur5_fast_v1/15000')
   